In [1]:
import os
os.getcwd()

'C:\\Users\\TWH'

In [2]:
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    mean_absolute_error, r2_score
)


In [3]:

# ==============================================================================
# 1. LOAD DATA & SELECT COMMON FEATURE SET
# ==============================================================================
df = pd.read_csv("data/processed/cleaned_traffic_features.csv")

feature_cols = [
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
    'is_weekend', 'is_holiday',
    'is_severe_weather', 'is_low_visibility',
    'temp_scaled', 'clouds_scaled'
]

X = df[feature_cols]
y_cls = df['high_risk']           # Target for Classification
y_reg = df['traffic_volume']       # Target for Regression

print(f"Feature matrix shape: {X.shape}")


Feature matrix shape: (48187, 10)


In [4]:

# ==============================================================================
# 2. TRAIN / TEST SPLITS
# ==============================================================================
# Stratified split for Classification to preserve high_risk class ratio
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X, y_cls, test_size=0.2, random_state=42, stratify=y_cls
)

# Standard split for Regression
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)


In [5]:
# ==============================================================================
# 3. TASK 1A: CLASSIFICATION MODELS (Accident Risk Prediction)
# ==============================================================================
print("\n--- Training Classification Models ---")

# Model 1: Logistic Regression (Baseline)
cls_lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
cls_lr.fit(X_train_c, y_train_c)
y_pred_lr = cls_lr.predict(X_test_c)
y_prob_lr = cls_lr.predict_proba(X_test_c)[:, 1]

# Model 2: Random Forest Classifier (Ensemble)
cls_rf = RandomForestClassifier(
    n_estimators=100, max_depth=12, class_weight='balanced', random_state=42, n_jobs=-1
)
cls_rf.fit(X_train_c, y_train_c)
y_pred_rf = cls_rf.predict(X_test_c)
y_prob_rf = cls_rf.predict_proba(X_test_c)[:, 1]

# Compile Classification Metrics
cls_metrics = pd.DataFrame([
    {
        'Model': 'Logistic Regression (Baseline)',
        'Accuracy': accuracy_score(y_test_c, y_pred_lr),
        'Precision': precision_score(y_test_c, y_pred_lr),
        'Recall': recall_score(y_test_c, y_pred_lr),
        'F1-Score': f1_score(y_test_c, y_pred_lr),
        'ROC AUC': roc_auc_score(y_test_c, y_prob_lr)
    },
    {
        'Model': 'Random Forest Classifier (Ensemble)',
        'Accuracy': accuracy_score(y_test_c, y_pred_rf),
        'Precision': precision_score(y_test_c, y_pred_rf),
        'Recall': recall_score(y_test_c, y_pred_rf),
        'F1-Score': f1_score(y_test_c, y_pred_rf),
        'ROC AUC': roc_auc_score(y_test_c, y_prob_rf)
    }
])


--- Training Classification Models ---


In [6]:
# ==============================================================================
# 4. TASK 1B: REGRESSION MODELS (Traffic Volume Prediction)
# ==============================================================================
print("--- Training Regression Models ---")

# Model 1: Ridge Regression (Baseline)
reg_ridge = Ridge(alpha=1.0)
reg_ridge.fit(X_train_r, y_train_r)
y_pred_ridge = reg_ridge.predict(X_test_r)

# Model 2: Random Forest Regressor (Ensemble)
reg_rf = RandomForestRegressor(n_estimators=100, max_depth=14, random_state=42, n_jobs=-1)
reg_rf.fit(X_train_r, y_train_r)
y_pred_reg_rf = reg_rf.predict(X_test_r)

# Compile Regression Metrics
reg_metrics = pd.DataFrame([
    {
        'Model': 'Ridge Regression (Baseline)',
        'MAE (Vehicles/hr)': mean_absolute_error(y_test_r, y_pred_ridge),
        'R-squared (R²)': r2_score(y_test_r, y_pred_ridge)
    },
    {
        'Model': 'Random Forest Regressor (Ensemble)',
        'MAE (Vehicles/hr)': mean_absolute_error(y_test_r, y_pred_reg_rf),
        'R-squared (R²)': r2_score(y_test_r, y_pred_reg_rf)
    }
])

--- Training Regression Models ---


In [7]:
# ==============================================================================
# 5. DISPLAY EVALUATION RESULTS
# ==============================================================================
print("\n" + "="*70)
print("CLASSIFICATION MODEL PERFORMANCE SUMMARY")
print("="*70)
print(cls_metrics.to_string(index=False))

print("\n" + "="*70)
print("REGRESSION MODEL PERFORMANCE SUMMARY")
print("="*70)
print(reg_metrics.to_string(index=False))


CLASSIFICATION MODEL PERFORMANCE SUMMARY
                              Model  Accuracy  Precision   Recall  F1-Score  ROC AUC
     Logistic Regression (Baseline)  0.965138   0.772069 0.980699  0.863968 0.993366
Random Forest Classifier (Ensemble)  0.986615   0.915872 0.970588  0.942436 0.997607

REGRESSION MODEL PERFORMANCE SUMMARY
                             Model  MAE (Vehicles/hr)  R-squared (R²)
       Ridge Regression (Baseline)         835.002904        0.708601
Random Forest Regressor (Ensemble)         265.382462        0.946874


### CLASSIFICATION MODEL PERFORMANCE SUMMARY###  
Massive False Positive Reduction: Logistic Regression's low precision ($77.21\%$) means that roughly $23\%$ of its "high-risk" flags were false alarms.  
Random Forest boosts precision to $91.59\%$, drastically reducing false alerts while maintaining high reliability.  

Balanced Detection: Although Logistic Regression catches marginally more true high-risk events (Recall of $98.07\%$ vs. $97.06\%$), Random Forest's high F1-Score ($0.9424$) makes it the vastly superior production choice by eliminating "alert fatigue."  

### REGRESSION MODEL PERFORMANCE SUMMARY  
Substantial Error Reduction: The Random Forest Regressor drops the average prediction error from $835$ vehicles/hour down to just $265$ vehicles/hour. On a highway corridor, an error of $835\text{ veh/hr}$ can hide an entire lane's capacity, whereas $265\text{ veh/hr}$ provides tight, actionable volume estimates.  

Variance Captured ($R^2$): Ridge Regression accounts for only $70.86\%$ of traffic variance due to its linear restriction. Random Forest captures $94.69\%$ of total variance, successfully modeling complex time-of-day curves, weekend variations, and weather shocks.  


### Deploy the Random Forest Ensemble models for both services:  
1. Volume Forecasting: Use Random Forest Regressor ($MAE = 265.38$, $R^2 = 0.9469$) to drive hourly travel timing recommendations.
2. Safety Alerting: Use Random Forest Classifier ($F1 = 0.9424$, $ROC\ AUC = 0.9976$) to flag high-risk corridor segments with minimal false positives.